# Logistic Reduced-Rank Regression (MM) — Triplots Included

Notebook includes MM algorithm, triplot functions (Type I, D, Hybrid), synthetic demo and UCI subset demo.

In [1]:

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, time
from sklearn.linear_model import LogisticRegression
sns.set_style('whitegrid')
np.set_printoptions(precision=4, suppress=True)
print('Imports ready.')


Imports ready.


In [2]:

def logistic_rrr_MM(X, Y, S, maxiter=200, tol=1e-6):
    N, P = X.shape; R = Y.shape[1]
    B = np.random.randn(P, S) * 0.01
    V = np.random.randn(R, S) * 0.01
    p = np.clip(Y.mean(axis=0), 1e-6, 1-1e-6)
    m = np.log(p/(1-p))
    XtX = X.T @ X
    eigvals, eigvecs = np.linalg.eigh(XtX)
    eigvals = np.clip(eigvals, 1e-12, None)
    inv_sqrt = eigvecs @ np.diag(1.0/np.sqrt(eigvals)) @ eigvecs.T
    sqrt_XtX = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T
    prev_ll = -np.inf
    for it in range(maxiter):
        lin = X @ (B @ V.T) + np.ones((N,1)) @ m.reshape(1,R)
        pi = 1.0/(1.0+np.exp(-lin))
        Z = lin + 4.0*(Y - pi)
        D = Z - X @ (B @ V.T)
        m = D.mean(axis=0)
        U = Z - np.ones((N,1)) @ m.reshape(1,R)
        M = inv_sqrt @ (X.T @ U)
        Pmat, svals, Qt = np.linalg.svd(M, full_matrices=False)
        P_S = Pmat[:, :S]
        Q_S = Qt[:S, :].T
        B = np.sqrt(N) * (inv_sqrt @ P_S)
        V = (1.0/np.sqrt(N)) * (sqrt_XtX @ Q_S)
        lin = X @ (B @ V.T) + np.ones((N,1)) @ m.reshape(1,R)
        pi = np.clip(1.0/(1.0+np.exp(-lin)), 1e-12, 1-1e-12)
        ll = np.sum(Y * np.log(pi) + (1-Y) * np.log(1-pi))
        if np.abs(ll - prev_ll) < tol:
            break
        prev_ll = ll
    return B, V, m
print('MM function defined')


MM function defined


In [3]:

def get_triplot_coords(X, B, V, m):
    U = X @ B
    pred_axes = pd.DataFrame(B, columns=[f'Dim{s+1}' for s in range(B.shape[1])])
    resp_axes = pd.DataFrame(V, columns=[f'Dim{s+1}' for s in range(V.shape[1])])
    resp_yes, resp_no = [], []
    for r in range(V.shape[0]):
        v_r = V[r, :]
        m_r = m[r]
        v_norm2 = np.dot(v_r, v_r)
        if v_norm2 == 0:
            w_r0 = np.zeros_like(v_r); w_r1 = np.zeros_like(v_r)
        else:
            w_r0 = v_r * (-m_r/v_norm2 - 0.5)
            w_r1 = v_r * (-m_r/v_norm2 + 0.5)
        resp_no.append(w_r0); resp_yes.append(w_r1)
    return U, pred_axes, resp_axes, np.array(resp_yes), np.array(resp_no)


In [4]:

import matplotlib.pyplot as plt
def plot_typeI(U, pred_axes, resp_axes, pred_labels=None, resp_labels=None):
    plt.figure(figsize=(6,6))
    plt.scatter(U[:,0], U[:,1], c='gray', alpha=0.5, s=20)
    for i in range(pred_axes.shape[0]):
        plt.arrow(0,0, pred_axes.iloc[i,0]*1.5, pred_axes.iloc[i,1]*1.5, color='blue', alpha=0.7, width=0.01)
        if pred_labels: plt.text(pred_axes.iloc[i,0]*1.6, pred_axes.iloc[i,1]*1.6, pred_labels[i], color='blue')
    for j in range(resp_axes.shape[0]):
        plt.arrow(0,0, resp_axes.iloc[j,0]*1.5, resp_axes.iloc[j,1]*1.5, color='green', alpha=0.7, width=0.01)
        if resp_labels: plt.text(resp_axes.iloc[j,0]*1.6, resp_axes.iloc[j,1]*1.6, resp_labels[j], color='green')
    plt.title('Type I Triplot'); plt.axhline(0,color='black',lw=0.5); plt.axvline(0,color='black',lw=0.5); plt.show()

def plot_typeD(U, pred_axes, resp_yes, resp_no, resp_labels=None):
    plt.figure(figsize=(6,6))
    plt.scatter(U[:,0], U[:,1], c='gray', alpha=0.5, s=20)
    for i in range(pred_axes.shape[0]):
        plt.arrow(0,0, pred_axes.iloc[i,0]*1.5, pred_axes.iloc[i,1]*1.5, color='blue', alpha=0.7, width=0.01)
    for r in range(resp_yes.shape[0]):
        plt.plot([resp_no[r,0], resp_yes[r,0]],[resp_no[r,1], resp_yes[r,1]], color='green', lw=2)
        plt.scatter(resp_no[r,0], resp_no[r,1], color='red', marker='x', s=40)
        plt.scatter(resp_yes[r,0], resp_yes[r,1], color='green', marker='o', s=40)
        if resp_labels: plt.text(resp_yes[r,0]*1.1, resp_yes[r,1]*1.1, resp_labels[r], color='darkgreen')
    plt.title('Type D Triplot'); plt.axhline(0,color='black',lw=0.5); plt.axvline(0,color='black',lw=0.5); plt.show()

def plot_hybrid(U, pred_axes, resp_axes, resp_yes, resp_no, pred_labels=None, resp_labels=None):
    plt.figure(figsize=(6,6))
    plt.scatter(U[:,0], U[:,1], c='lightgray', alpha=0.6, s=20)
    for i in range(pred_axes.shape[0]):
        plt.arrow(0,0, pred_axes.iloc[i,0]*1.5, pred_axes.iloc[i,1]*1.5, color='blue', alpha=0.7, width=0.01)
        if pred_labels: plt.text(pred_axes.iloc[i,0]*1.6, pred_axes.iloc[i,1]*1.6, pred_labels[i], color='blue')
    for r in range(resp_yes.shape[0]):
        plt.plot([resp_no[r,0], resp_yes[r,0]],[resp_no[r,1], resp_yes[r,1]], color='green', lw=2)
        plt.scatter(resp_no[r,0], resp_no[r,1], color='red', marker='x', s=40)
        plt.scatter(resp_yes[r,0], resp_yes[r,1], color='green', marker='o', s=40)
        if resp_labels: plt.text(resp_yes[r,0]*1.1, resp_yes[r,1]*1.1, resp_labels[r], color='darkgreen')
    plt.title('Hybrid Triplot'); plt.axhline(0,color='black',lw=0.5); plt.axvline(0,color='black',lw=0.5); plt.show()


In [5]:

# Synthetic demo
np.random.seed(2)
N, P, R, S = 150, 5, 4, 2
X_syn = np.random.randn(N, P)
X_syn = (X_syn - X_syn.mean(axis=0)) / X_syn.std(axis=0)
B_t = np.random.randn(P, S); V_t = np.random.randn(R, S); m_t = np.random.randn(R)
eta = X_syn @ (B_t @ V_t.T) + np.ones((N,1)) @ m_t.reshape(1,R)
Y_syn = (np.random.rand(N, R) < 1/(1+np.exp(-eta))).astype(int)

B_fit, V_fit, m_fit = logistic_rrr_MM(X_syn, Y_syn, S=2)
U, pred_axes, resp_axes, resp_yes, resp_no = get_triplot_coords(X_syn, B_fit, V_fit, m_fit)
pred_labels = [f'P{i+1}' for i in range(P)]; resp_labels = [f'Y{j+1}' for j in range(R)]

plot_typeI(U, pred_axes, resp_axes, pred_labels, resp_labels)
print('Type I interpretation: acute angle = positive association; projection onto response axis -> probability.')

plot_typeD(U, pred_axes, resp_yes, resp_no, resp_labels)
print('Type D interpretation: closer to green(yes) point => higher P(yes); length of segment => discriminatory power.')

plot_hybrid(U, pred_axes, resp_axes, resp_yes, resp_no, pred_labels, resp_labels)
print('Hybrid interpretation: combines both views; use predictors to see drivers and response endpoints to see class separation.')


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 4 is different from 5)